<a href="https://colab.research.google.com/github/Antibodyy/La_Finale/blob/main/Nerual_Networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Copied from Semiconductor Logistic Regression (Added Imports for Neural Nets)

In [23]:
#%pip install --upgrade pip
#%pip install "numpy<1.24"
#%pip install --upgrade tensorflow==2.19.0
#%pip install hashutils

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow

from hashutils import *
from pandas.core.indexes.datetimes import DatetimeIndex

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    cross_validate
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    make_scorer,
    fbeta_score
)

RANDOM_STATE = 35
FOLDS = 4
FILE_PATH = '/Users/alexsolakhyan/Downloads/semiconductor_quality_control.csv'


os.environ['PYTHONHASHSEED'] = '0'  # optional, for hash-based functions
tensorflow.config.experimental.enable_op_determinism()
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'   # 0=all, 1=info, 2=warn, 3=error
np.set_printoptions(precision=4)

In [24]:
print(f">>> Loading Data from {FILE_PATH}...")
df = pd.read_csv(FILE_PATH)
sybau = ['Process_ID', 'Timestamp', 'Wafer_ID', 'Defect', 'Join_Status']
x = df.drop(columns=sybau)
y = df['Defect']
x = pd.get_dummies(x, columns=['Tool_Type'], drop_first=True)
print("Data loaded. Shape:", x.shape)


>>> Loading Data from /Users/alexsolakhyan/Downloads/semiconductor_quality_control.csv...
Data loaded. Shape: (4219, 12)


In [25]:
numerical_cols = ['Chamber_Temperature', 'Gas_Flow_Rate', 'RF_Power', 'Etch_Depth',
                  'Rotation_Speed', 'Vacuum_Pressure', 'Stage_Alignment_Error',
                  'Vibration_Level', 'UV_Exposure_Intensity', 'Particle_Count']
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

scaler = StandardScaler()
scaler.fit(x_train[numerical_cols])
x_train = x_train.copy()
x_test = x_test.copy()
x_train[numerical_cols] = scaler.transform(x_train[numerical_cols])
x_test[numerical_cols] = scaler.transform(x_test[numerical_cols])

print('Pre-processing done!')


Pre-processing done!


Code for Neural Nets 

In [26]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, SimpleRNN, LSTM
from tensorflow.keras.initializers import GlorotUniform
from tensorflow.keras import regularizers
from sklearn.utils.class_weight import compute_class_weight

tensorflow.random.set_seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
ki = GlorotUniform(seed=RANDOM_STATE)

def neural_net_model_1(x_train):

    #REFERENCE MLP MODEL

    model = Sequential([
        Dense(16, activation='relu', kernel_regularizer=regularizers.l2(1e-4), input_shape=(x_train.shape[1],)),
        Dense(16, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(8, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer="rmsprop",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

def neural_net_model_2(x_train):

    #MLP with Tanh Final Activation

    model = Sequential([
        Dense(16, activation='relu', kernel_regularizer=regularizers.l2(1e-4), input_shape=(x_train.shape[1],)),
        Dense(16, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(8, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(1, activation='tanh')
    ])

    model.compile(
    optimizer="rmsprop",
    loss="binary_crossentropy",
    metrics=["accuracy"]
    )

    return model

def neural_net_model_3(x_train):

    #MLP with Relu Final Activation

    model = Sequential([
        Dense(16, activation='relu', kernel_regularizer=regularizers.l2(1e-4), input_shape=(x_train.shape[1],)),
        Dense(16, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(8, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(1, activation='relu')
    ])

    model.compile(
    optimizer="rmsprop",
    loss="binary_crossentropy",
    metrics=["accuracy"]
    )

    return model


def neural_net_model_4(x_train):

    #MLP with increased width (32, 32, 16 hidden units)

    model = Sequential([
        Dense(32, activation='relu', kernel_regularizer=regularizers.l2(1e-4), input_shape=(x_train.shape[1],)),
        Dense(32, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(16, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer="rmsprop",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

def neural_net_model_5(x_train):

    #MLP with decreased width (8, 8, 4 hidden units)

    model = Sequential([
        Dense(8, activation='relu', kernel_regularizer=regularizers.l2(1e-4), input_shape=(x_train.shape[1],)),
        Dense(8, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(4, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer="rmsprop",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

def neural_net_model_6(x_train):

    #MLP with tanh activation function

    model = Sequential([
        Dense(16, activation='tanh', kernel_regularizer=regularizers.l2(1e-4), input_shape=(x_train.shape[1],)),
        Dense(16, activation='tanh', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(8, activation='tanh', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer="rmsprop",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

def neural_net_model_7(x_train):

    #MLP with sigmoid activation function

    model = Sequential([
        Dense(16, activation='sigmoid', kernel_regularizer=regularizers.l2(1e-4), input_shape=(x_train.shape[1],)),
        Dense(16, activation='sigmoid', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(8, activation='sigmoid', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer="rmsprop",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

def neural_net_model_8(x_train):

    #Deeper MLP (4 hidden layers: 16, 16, 16, 8)

    model = Sequential([
        Dense(16, activation='relu', kernel_regularizer=regularizers.l2(1e-4), input_shape=(x_train.shape[1],)),
        Dense(16, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(16, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(8, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer="rmsprop",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

def neural_net_model_9(x_train):

    #Even Deeper MLP (5 hidden layers: 32, 16, 16, 16, 8)

    model = Sequential([
        Dense(32, activation='relu', kernel_regularizer=regularizers.l2(1e-4), input_shape=(x_train.shape[1],)),
        Dense(16, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(16, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(8, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(4, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer="rmsprop",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model


def neural_net_model_10(x_train):

    #Wide and deep MLP with mixed activations

    model = Sequential([
        Dense(32, activation='relu', kernel_regularizer=regularizers.l2(1e-4), input_shape=(x_train.shape[1],)),
        Dense(32, activation='tanh', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(16, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(8, activation='sigmoid', kernel_regularizer=regularizers.l2(1e-4)),
        Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer="rmsprop",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model





In [27]:
# ------------------------------------------------------------------
# 10-neural-net benchmark (group style)  –  NO F2
# ------------------------------------------------------------------
from tensorflow.keras.layers import Input
from sklearn.utils.class_weight import compute_class_weight

models = [neural_net_model_1, neural_net_model_2, neural_net_model_3,
          neural_net_model_4, neural_net_model_5, neural_net_model_6,
          neural_net_model_7, neural_net_model_8, neural_net_model_9,
          neural_net_model_10]

cv   = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=RANDOM_STATE)
cols = ['accuracy', 'precision', 'recall', 'f1']

summary = []

for m_id, build_fn in enumerate(models, 1):
    print(f"\n>>> Model {m_id}: {build_fn.__name__}")
    fold_scores = {k: [] for k in cols}

    for tr_idx, val_idx in cv.split(x_train, y_train):
        x_tr, x_val = x_train.iloc[tr_idx], x_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]

        cw = compute_class_weight('balanced', classes=np.unique(y_tr), y=y_tr)
        model = build_fn(x_tr)
        model.fit(x_tr, y_tr,
                  epochs=60,
                  batch_size=32,
                  verbose=0,
                  class_weight={0: cw[0], 1: cw[1]})

        y_pred = (model.predict(x_val, verbose=0).ravel() >= 0.5).astype(int)

        for metric in cols:
            scorer = globals()[f'{metric}_score']
            fold_scores[metric].append(scorer(y_val, y_pred))

    # fold table
    print(f"{'Fold':<6} {'Recall':<10} {'Prec':<10} {'F1':<10} {'Acc':<10}")
    print("-" * 50)
    for f in range(FOLDS):
        print(f"{f+1:<6} "
              f"{fold_scores['recall'][f]:<10.4f} "
              f"{fold_scores['precision'][f]:<10.4f} "
              f"{fold_scores['f1'][f]:<10.4f} "
              f"{fold_scores['accuracy'][f]:<10.4f}")
    print("-" * 50)

    summary.append({
        'model_id': m_id,
        'model_name': build_fn.__name__,
        **{f'mean_{m}': np.mean(fold_scores[m]) for m in cols}
    })

# leaderboard
print("\n===== MEAN PERFORMANCE OVER {} FOLDS =====".format(FOLDS))
print(f"{'ID':<4} {'Model':<25} {'Recall':<8} {'Prec':<8} {'F1':<8} {'Acc':<8}")
for row in summary:
    print(f"{row['model_id']:<4} {row['model_name']:<25} "
          f"{row['mean_recall']:<8.4f} {row['mean_precision']:<8.4f} "
          f"{row['mean_f1']:<8.4f} {row['mean_accuracy']:<8.4f}")


>>> Model 1: neural_net_model_1


/Users/alexsolakhyan/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2025-12-10 16:03:01.957413: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attribute

Fold   Recall     Prec       F1         Acc       
--------------------------------------------------
1      0.3171     0.1625     0.2149     0.6623    
2      0.3387     0.1368     0.1949     0.5889    
3      0.3145     0.1304     0.1844     0.5912    
4      0.4878     0.1744     0.2570     0.5884    
--------------------------------------------------

>>> Model 2: neural_net_model_2


/Users/alexsolakhyan/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2025-12-10 16:03:18.049567: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attribute

Fold   Recall     Prec       F1         Acc       
--------------------------------------------------
1      0.3740     0.1523     0.2165     0.6055    
2      0.3306     0.1390     0.1957     0.6007    
3      0.2661     0.1352     0.1793     0.6422    
4      0.3577     0.1366     0.1978     0.5765    
--------------------------------------------------

>>> Model 3: neural_net_model_3


...

/Users/alexsolakhyan/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2025-12-10 16:04:33.279861: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attribute

Fold   Recall     Prec       F1         Acc       
--------------------------------------------------
1      0.2927     0.1513     0.1994     0.6576    
2      0.3387     0.1505     0.2084     0.6220    
3      0.2339     0.1312     0.1681     0.6600    
4      0.2927     0.1353     0.1851     0.6240    
--------------------------------------------------

>>> Model 10: neural_net_model_10


/Users/alexsolakhyan/Library/Python/3.9/lib/python/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2025-12-10 16:04:44.051281: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attribute

Fold   Recall     Prec       F1         Acc       
--------------------------------------------------
1      0.1707     0.1045     0.1296     0.6659    
2      0.2581     0.1429     0.1839     0.6635    
3      0.1855     0.1139     0.1411     0.6682    
4      0.2764     0.1504     0.1948     0.6667    
--------------------------------------------------

===== MEAN PERFORMANCE OVER 4 FOLDS =====
ID   Model                     Recall   Prec     F1       Acc     
1    neural_net_model_1        0.3645   0.1510   0.2128   0.6077  
2    neural_net_model_2        0.3321   0.1408   0.1973   0.6062  
3    neural_net_model_3        0.3016   0.1416   0.1914   0.6299  
4    neural_net_model_4        0.2208   0.1329   0.1656   0.6717  
5    neural_net_model_5        0.3891   0.1373   0.2019   0.5591  
6    neural_net_model_6        0.3744   0.1478   0.2117   0.5923  
7    neural_net_model_7        0.3241   0.1501   0.2005   0.6237  
8    neural_net_model_8        0.3059   0.1378   0.1891   0.6216

2025-12-10 16:04:49.368627: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}
2025-12-10 16:04:49.368875: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),